<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: -8px;">
    <div style="width: 50%;">
        <h2>
            <span style="color: #B30033;">▍</span>TRABAJO DE FIN DE MÁSTER
        </h2>
        <h1 style="margin-top: -10px;">
            Creación de un framework para recomendar las personas más adecuadas para trabajar en un determinado proyecto de ingeniería del software
        </h1>
    </div>
    <div style="width: 50%; text-align: right;">
        <div style="display: flex; justify-content: space-between; align-items: flex-start; margin-top: 30px;">
            <div style="width: 20%;"></div>
            <div style="width: 80%; border-left: 2px solid #555555; padding-left: 20px;">
                <div style="margin-bottom: 20px;">
                    <p style="margin: 0; font-size: 1.4em; font-weight: bold;">
                        Profesores
                    </p>
                </div>
                <div style="margin-top: 8px; text-align: right;">
                    <span style="display: block; font-size: 1em; color: #D0D0D0;">
                        Aurora Vizcaíno Barcelo
                    </span>
                    <span style="display: block; font-size: 1em; color: #D0D0D0; margin-top: 5px;">
                        Julio Alberto López Gómez
                    </span>
                </div>
            </div>
        </div>
    </div>
</div>

<div style="margin-top: 30px; padding: 20px; background-color: rgba(179, 0, 51, 0.05); border-left: 4px solid #B30033; border-radius: 4px;">
    <h3 style="margin-top: 0; color: #B30033;">🎯 Objetivo de esta Libreta (Fase 4: Inferencia y Casos de Uso)</h3>
    <p style="font-size: 1.1em; line-height: 1.6; color: #E0E0E0;">
        En esta libreta culmina el desarrollo técnico del proyecto <strong>NetSTAFF</strong>. Tras haber procesado los datos brutos de <em>Stack Overflow</em> y haber entrenado con éxito nuestra red neuronal simétrica (Autoencoder) en las libretas anteriores, el propósito aquí <strong>es obtener resultados con una prueba realista</strong>.
    </p>
    <ul style="font-size: 1.1em; line-height: 1.6; color: #D0D0D0; margin-bottom: 0;">
        <li><strong>Generación del Espacio Latente:</strong> Cargaremos los pesos óptimos del modelo para realizar la inferencia y extraer el "ADN técnico" (vector de 16 dimensiones) de nuestros +35k candidatos.</li>
        <li><strong>Motor de Recomendación:</strong> Implementaremos el algoritmo de <em>Matching</em> basado en la <strong>Similitud del Coseno</strong> para medir la distancia semántica y geométrica entre ofertas y desarrolladores.</li>
        <li><strong>Filtro Híbrido de RRHH:</strong> Demostraremos la utilidad real del sistema combinando reglas estrictas de negocio (ej. años de experiencia) con Inteligencia Artificial mediante casos de uso reales.</li>
    </ul>
</div>
<br/>

<div style="border-bottom: 2px solid #555555; padding-bottom: 25px; margin-bottom: 10px">
    <div style="display: flex; align-items: center; margin-bottom: 10px;">
        <span style="color: #B30033; font-size: 1.5em; margin-right: 10px;">▍</span>
        <h3 style="margin: 0; font-size: 1.4em; font-weight: bold">
            Realizado por Miguel Ángel Sánchez-Medina Camacho <span style="font-size: 0.7em; font-style: italic; font-weight: normal;"></span>
        </h3>
    </div>
</div>

### 4.1. Generación del Espacio Latente (Inferencia)

En esta primera fase lo que hacemos es cargar nuestra matriz de datos preprocesados y el **Autoencoder** ya entrenado con sus pesos óptimos de la anterior libreta.

El objetivo aquí es realizar la **inferencia**. Para ello, pasaremos a los más de 35.000 candidatos por el *Encoder* para comprimir sus 217 variables técnicas originales en un vector denso de 16 dimensiones. Esta nueva matriz resultante será nuestra **Base de Datos Latente**, que contiene el "ADN técnico" puro de cada desarrollador, libre de ruido.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(os.path.abspath(os.path.join('../..')))
from src.hard_skills import config, model_utils
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df_model = pd.read_csv(config.MODEL_MATRIX_PATH, index_col='CandidateId')
X_tensor = torch.tensor(df_model.values, dtype=torch.float32).to(device)

input_dim = df_model.shape[1]
modelo = model_utils.CandidateAutoencoder(input_dim=input_dim, latent_dim=16).to(device)
ruta_modelo = os.path.join('../..', 'models', 'autoencoder_candidatos.pth')

modelo.load_state_dict(torch.load(ruta_modelo, map_location=device))
modelo.eval()

with torch.no_grad():
    perfiles_latentes = modelo.get_latent_profile(X_tensor)

df_latente = pd.DataFrame(
    perfiles_latentes.cpu().numpy(), 
    index=df_model.index,            
    columns=[f'ADN_{i+1}' for i in range(16)]
)

print(f"Dimensiones antes: {df_model.shape}")
print(f"Dimensiones después: {df_latente.shape}")
display(df_latente.head(3))

Dimensiones antes: (35288, 217)
Dimensiones después: (35288, 16)


,ADN_1,ADN_2,ADN_3,ADN_4,ADN_5,ADN_6,ADN_7,ADN_8,ADN_9,ADN_10,ADN_11,ADN_12,ADN_13,ADN_14,ADN_15,ADN_16
CandidateId,,,,,,,,,,,,,,,,
0,6.606681,-0.550827,3.405831,1.063399,0.299338,0.526119,1.493528,4.998232,3.472701,-1.066841,5.756108,-3.120936,1.362790,-0.132461,-3.974200,0.803918
1,5.187068,-0.182071,3.285441,2.013331,-0.350938,2.827365,1.451458,3.827067,3.351021,0.758818,4.212327,-3.202677,1.881642,-1.206098,-5.293142,2.154918
2,7.796721,0.641201,2.609481,-0.062276,0.772495,-0.514932,-1.431648,2.738874,1.506030,0.130791,4.358593,-6.779866,-1.074692,0.637684,-4.046515,-0.712095


### 4.2. Motor de Recomendación: Similitud del Coseno

Aquí vamos a definir el núcleo matemático de NetSTAFF. La función `recomendar_candidatos` no busca coincidencias de texto, sino que realiza una **comparación geométrica**.

Dado una oferta de trabajo, el algoritmo calcula la **Similitud del Coseno** frente a todos los perfiles de nuestra Base de Datos Latente. Al medir el ángulo entre los vectores (donde 0º es un match del 100%), priorizamos que el candidato tenga la misma arquitectura de conocimiento y ecosistema tecnológico que la oferta, sin importar si no coinciden las palabras exactas.

In [2]:
def recomendar_candidatos(vector_proyecto_latente, df_candidatos_latentes, top_n=5):
    matriz_candidatos = df_candidatos_latentes.values
    vector_proyecto = vector_proyecto_latente.reshape(1,-1)
    similitudes = cosine_similarity(vector_proyecto, matriz_candidatos)[0]
    df_resultados = df_candidatos_latentes.copy()
    df_resultados['Similitud'] = similitudes
    
    df_resultados = df_resultados.sort_values(by='Similitud', ascending=False)
    return df_resultados.head(top_n)

print("Función de recomendación lista")

Función de recomendación lista


### 4.3. Prueba de Pipeline (Inferencia Base)

Antes de pasar a la demo donde trataremos de hacer un uso realista del sistema, aquí vamos a validar el flujo completo de la arquitectura. 
Para ello, creamos un "candidato fantasma" inicial (un vector de 217 ceros), lo convertimos a su representación latente de 16 dimensiones pasando por la red neuronal, y comprobamos que el motor de recomendación es capaz de devolver y ordenar un ranking de afinidad correctamente.

In [3]:
proyecto_ideal = pd.DataFrame(np.zeros((1, df_model.shape[1])), columns=df_model.columns)
tensor_proyecto = torch.tensor(proyecto_ideal.values, dtype=torch.float32).to(device)

with torch.no_grad():
    adn_proyecto = modelo.get_latent_profile(tensor_proyecto).cpu().numpy()

print("Buscando a los mejores candidatos en el Espacio Latente...")
top_candidatos = recomendar_candidatos(adn_proyecto, df_latente, top_n=5)

print("\nTOP 5 CANDIDATOS ENCONTRADOS")
display(top_candidatos)

Buscando a los mejores candidatos en el Espacio Latente...

TOP 5 CANDIDATOS ENCONTRADOS


,ADN_1,ADN_2,ADN_3,ADN_4,ADN_5,ADN_6,ADN_7,ADN_8,ADN_9,ADN_10,ADN_11,ADN_12,ADN_13,ADN_14,ADN_15,ADN_16,Similitud
CandidateId,,,,,,,,,,,,,,,,,
34391,4.218367,-1.454255,3.107139,-1.816376,1.564334,0.443933,-0.396399,2.701026,-0.131831,2.676277,3.751944,-3.092474,-0.013347,1.322381,-2.540063,0.436486,0.998707
35139,4.251938,-1.550600,3.326034,-2.015644,1.538679,0.367309,-0.293733,2.939116,-0.196272,2.811812,3.901959,-3.425063,0.104097,1.418041,-2.596242,0.388728,0.998345
19267,4.661568,-1.782325,3.757460,-2.154347,1.516239,0.409245,-0.229295,2.979042,-0.073640,2.984511,4.352895,-3.898723,0.075619,1.381519,-2.721747,0.271802,0.998122
8747,3.937645,-1.407168,2.975058,-1.849189,1.471472,0.371823,-0.277257,2.556269,-0.256296,2.617633,3.520177,-2.999614,-0.019399,1.364466,-2.257307,0.348016,0.997937
19856,4.893656,-1.881917,4.020047,-2.386768,1.622524,0.413120,-0.275108,3.083513,-0.228744,3.304681,4.486679,-3.829716,-0.090226,1.403192,-2.702538,0.473543,0.997743


### 4.4. Demostración de Negocio - Caso de uso real

En este bloque simulamos un entorno real de selección de personal. Para que el sistema dé valor real a la empresa, implementamos un **Sistema Recomendador Híbrido** que actúa en dos fases:

1. **Filtro Rígido (Reglas de Negocio):** Aplicamos las exigencias innegociables de Recursos Humanos (por ejemplo, filtrar solo perfiles *Senior*).
2. **Filtro Semántico (Deep Learning):** Sobre los finalistas del paso anterior, nuestra Red Neuronal busca el candidato perfecto, ordenando a los candidatos por su proximidad angular a las tecnologías demandadas (Ej: *Python, SQL, AWS*).

*El resultado es un ranking 100% objetivo que detecta talento oculto valorando el ecosistema global del candidato.*


In [4]:
CASO_ACTUAL = 1
print(f"DEMOSTRACIÓN - CASO DE USO REAL #{CASO_ACTUAL}")
print("=" * 60)

if CASO_ACTUAL == 1:
    print(" OFERTA: Empresa FinTech busca Data Engineer / Backend")
    print(" REQUISITOS: Python, SQL, AWS, Experiencia Senior (>0.7)")

    molde = df_model[(df_model['YearsCode'] > 0.7) & 
                     (df_model.get('LanguageHaveWorkedWith_Python', 0) == 1.0) & 
                     (df_model.get('LanguageHaveWorkedWith_SQL', 0) == 1.0)].iloc[0]
    
    proyecto_ideal = pd.DataFrame([molde.values], columns=df_model.columns)
    
    if 'PlatformHaveWorkedWith_AWS' in proyecto_ideal.columns: 
        proyecto_ideal['PlatformHaveWorkedWith_AWS'] = 1.0

elif CASO_ACTUAL == 2:
    print(" OFERTA: Agencia de Marketing busca Desarrollador Frontend")
    print(" REQUISITOS: JavaScript, HTML/CSS, Experiencia Junior (<0.3)")
    
    molde = df_model[(df_model['YearsCode'] < 0.3) & 
                     (df_model.get('LanguageHaveWorkedWith_JavaScript', 0) == 1.0) & 
                     (df_model.get('LanguageHaveWorkedWith_HTML/CSS', 0) == 1.0)].iloc[0]
    
    proyecto_ideal = pd.DataFrame([molde.values], columns=df_model.columns)

print("=" * 60)

tensor_proyecto = torch.tensor(proyecto_ideal.values, dtype=torch.float32).to(device)
with torch.no_grad():
    adn_proyecto = modelo.get_latent_profile(tensor_proyecto).cpu().numpy()

if CASO_ACTUAL == 1:
    candidatos_validos = df_model[df_model['YearsCode'] >= 0.7].index
elif CASO_ACTUAL == 2:
    candidatos_validos = df_model[df_model['YearsCode'] <= 0.3].index
else:
    candidatos_validos = df_model.index
df_latente_filtrado = df_latente.loc[candidatos_validos]
top_candidatos = recomendar_candidatos(adn_proyecto, df_latente_filtrado, top_n=5)

print("\nTOP 5 CANDIDATOS:\n")
ids_ganadores = top_candidatos.index.tolist()
for i, cand_id in enumerate(ids_ganadores):
    similitud = top_candidatos.loc[cand_id, 'Similitud']
    print(f" RANKING #{i+1} | Perfil {chr(65+i)} | Nivel de Match: {similitud*100:.2f}%")
    perfil_real = df_model.loc[cand_id]
    
    tecnologias = [col for col in perfil_real.index if "Language" in col and perfil_real[col] == 1.0]
    plataformas = [col for col in perfil_real.index if "Platform" in col and perfil_real[col] == 1.0]
    experiencia = perfil_real.get('YearsCode', 'N/A')
    
    if isinstance(experiencia, float):
        experiencia = f"{experiencia:.2f} (Escalado)"
    
    print(f"    * Tecnologías clave: {', '.join([t.split('_')[-1] for t in tecnologias]) if tecnologias else 'No especificadas'}")
    print(f"    * Plataformas/Nube: {', '.join([p.split('_')[-1] for p in plataformas]) if plataformas else 'No especificadas'}")
    print(f"    * Nivel de Experiencia: {experiencia}")
    print("-" * 60)

DEMOSTRACIÓN - CASO DE USO REAL #1
 OFERTA: Empresa FinTech busca Data Engineer / Backend
 REQUISITOS: Python, SQL, AWS, Experiencia Senior (>0.7)

TOP 5 CANDIDATOS:

 RANKING #1 | Perfil A | Nivel de Match: 100.00%
    * Tecnologías clave: JavaScript, HTML/CSS, SQL, Python, Java, PowerShell, C
    * Plataformas/Nube: Stack Overflow, Reddit, Stack Exchange
    * Nivel de Experiencia: 0.77 (Escalado)
------------------------------------------------------------
 RANKING #2 | Perfil B | Nivel de Match: 99.06%
    * Tecnologías clave: JavaScript, HTML/CSS, SQL, Python
    * Plataformas/Nube: Stack Overflow, Stack Exchange
    * Nivel de Experiencia: 0.77 (Escalado)
------------------------------------------------------------
 RANKING #3 | Perfil C | Nivel de Match: 98.72%
    * Tecnologías clave: JavaScript, HTML/CSS, SQL, Python, Java, PowerShell, C++, C
    * Plataformas/Nube: Stack Overflow, Reddit, Stack Exchange
    * Nivel de Experiencia: 0.77 (Escalado)
-----------------------------